In [ ]:
"""
Concepts:
1. Ordered Boosting
2. Native Categorical Features
3. Target Leakage Prevention
4. Overfitting Detector
5. Feature Importance
6. SHAP Explainability
"""

# =====================================================
# IMPORTS
# =====================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split,cross_val_score,RandomizedSearchCV,learning_curve)
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix,classification_report,roc_curve,
    precision_recall_curve)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from catboost import CatBoostClassifier

In [ ]:
# =====================================================
# STEP 1 : DATA UNDERSTANDING
# =====================================================

data = load_breast_cancer()
df = pd.DataFrame(data.data,columns=data.feature_names)
df["target"] = data.target

print(df.head())
print(df.info())
print(df.describe())
print(df["target"].value_counts())

In [ ]:
# =====================================================
# STEP 2 : EDA
# =====================================================

print(df.isnull().sum())
print(df.duplicated().sum())

In [ ]:
# =====================================================
# STEP 3 : TRAIN / VALIDATION / TEST
# =====================================================

X = df.drop("target", axis=1)
y = df["target"]
X_temp, X_test, y_temp, y_test = train_test_split(X,y,test_size=0.15,random_state=42,stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp,y_temp,test_size=0.1765,random_state=42,stratify=y_temp)

In [ ]:
# =====================================================
# STEP 4 : PREPROCESSING
# =====================================================

# CatBoost does not require scaling

In [ ]:
# =====================================================
# STEP 5 : BASELINE MODEL
# =====================================================

cat = CatBoostClassifier(iterations=300,learning_rate=0.05,depth=6,eval_metric='AUC',random_seed=42,verbose=False)
cat.fit(X_train,y_train)

In [ ]:
# =====================================================
# STEP 6 : VALIDATION METRICS
# =====================================================

val_pred = cat.predict(X_val)
val_prob = cat.predict_proba(X_val)[:,1]
print("Accuracy:",accuracy_score(y_val,val_pred))
print("Precision:",precision_score(y_val,val_pred))
print("Recall:",recall_score(y_val,val_pred))
print("F1:",f1_score(y_val,val_pred))
print("ROC AUC:",roc_auc_score(y_val,val_prob))

In [ ]:
# =====================================================
# STEP 7 : CROSS VALIDATION
# =====================================================

cv_scores = cross_val_score(cat,X_train,y_train,cv=5,scoring='roc_auc',n_jobs=-1)
print(cv_scores.mean())
print(cv_scores.std())

In [ ]:
# =====================================================
# STEP 8 : HYPERPARAMETER TUNING
# =====================================================

param_dist = {
    "iterations":[200,300,500],
    "learning_rate":[0.01,0.05,0.1],
    "depth":[4,6,8,10],
    "l2_leaf_reg":[1,3,5,10],
    "border_count":[32,64,128]}

search = RandomizedSearchCV(CatBoostClassifier(eval_metric='AUC',verbose=False,random_seed=42),
                            param_dist,n_iter=25,scoring='roc_auc',cv=5,random_state=42,n_jobs=-1)
search.fit(X_train,y_train)
best_model = search.best_estimator_
print(search.best_params_)


In [ ]:
# =====================================================
# STEP 9 : VALIDATION RECHECK
# =====================================================

val_prob = best_model.predict_proba(X_val)[:,1]
val_auc = roc_auc_score(y_val,val_prob)
print("Validation AUC:", val_auc)

In [ ]:
# =====================================================
# STEP 10 : TRAIN VS VALIDATION
# =====================================================

train_prob = best_model.predict_proba(X_train)[:,1]
train_auc = roc_auc_score(y_train,train_prob)
print("Train AUC:",train_auc)
print("Validation AUC:",val_auc)

In [ ]:

# =====================================================
# STEP 11 : MODEL ANALYSIS
# =====================================================

# FEATURE IMPORTANCE

importance_df = pd.DataFrame({"Feature":X.columns,"Importance":best_model.feature_importances_})
importance_df = importance_df.sort_values(by="Importance",ascending=False)
print(importance_df.head(15))

plt.figure(figsize=(8,5))
plt.barh(importance_df["Feature"][:15],importance_df["Importance"][:15])
plt.title("Feature Importance")
plt.show()

# PERMUTATION IMPORTANCE

perm = permutation_importance(best_model,X_val,y_val,random_state=42)
print(perm.importances_mean)

# CONFUSION MATRIX

cm = confusion_matrix(y_val,best_model.predict(X_val))
sns.heatmap(cm,annot=True,fmt='d')
plt.show()

# ROC CURVE

fpr,tpr,_ = roc_curve(y_val,val_prob)
plt.plot(fpr,tpr)
plt.plot([0,1],[0,1])
plt.show()

# PR CURVE

precision, recall, _ = precision_recall_curve(y_val,val_prob)
plt.plot(recall,precision)
plt.show()

# CALIBRATION CURVE

prob_true, prob_pred = calibration_curve(y_val,val_prob,n_bins=10)
plt.plot(prob_pred,prob_true,marker='o')
plt.plot([0,1],[0,1])
plt.show()

# DEPTH ANALYSIS

depths = [4,6,8,10]
scores = []
for d in depths:
    model = CatBoostClassifier(depth=d,verbose=False,random_seed=42)
    model.fit(X_train,y_train)
    prob = model.predict_proba(X_val)[:,1]
    scores.append(roc_auc_score(y_val,prob))

plt.plot(depths,scores,marker='o')
plt.title("Depth Analysis")
plt.show()

# LEARNING CURVE

train_sizes, train_scores, val_scores = learning_curve(best_model,X_train,y_train,cv=5,scoring='roc_auc',n_jobs=-1)
plt.plot(train_sizes,np.mean(train_scores,axis=1),label='Train')
plt.plot(train_sizes,np.mean(val_scores,axis=1),label='Validation')

plt.legend()
plt.show()

In [ ]:
# =====================================================
# STEP 12 : FINAL MODEL
# =====================================================

final_model = best_model

In [ ]:
# =====================================================
# STEP 13 : TEST EVALUATION
# =====================================================

test_pred = final_model.predict(X_test)
test_prob = final_model.predict_proba(X_test)[:,1]
print(classification_report(y_test,test_pred))
print("Test ROC AUC:",roc_auc_score(y_test,test_prob))

In [ ]:
# =====================================================
# STEP 14 : DEPLOYMENT READINESS
# =====================================================

print("CV:",cv_scores.mean())
print("Validation:",val_auc)
print("Test:",roc_auc_score(y_test,test_prob))

# =====================================================
# STEP 15 : INTERVIEW QUESTIONS
# =====================================================

"""
1. What is CatBoost?
2. Why CatBoost was developed?
3. What is Ordered Boosting?
4. How does CatBoost prevent target leakage?
5. Difference between CatBoost and XGBoost?
6. Difference between CatBoost and LightGBM?
7. What is l2_leaf_reg?
8. What is border_count?
9. Why CatBoost handles categorical features well?
10. Why does CatBoost require less preprocessing?
11. What is overfitting detector?
12. Why CatBoost performs well on small datasets?
13. Why scaling is unnecessary?
14. What is feature importance?
15. What is permutation importance?
16. What is calibration curve?
17. Why ROC-AUC is preferred?
18. When should CatBoost be preferred?
19. CatBoost advantages and limitations?
20. Explain CatBoost end-to-end.
"""